In [1]:
from pathlib import Path
import sys, re
PROJECT = next((p / "homework_8" for p in (Path.cwd(), *Path.cwd().parents) if (p / "homework_8/captures").exists()), Path.cwd() / "homework_8")
sys.path[:0] = [str(PROJECT), *map(str, PROJECT.glob(".venv/lib/python*/site-packages"))]
from scapy_lab.http_analysis import read_pcap, build_transactions
from scapy_lab.xss import analyze_transactions, build_comparisons
PCAP = PROJECT / "captures/Чарыков_Данила_ДЗ5.pcap"
print(PCAP)

/home/socrates/university/homework_8/captures/Чарыков_Данила_ДЗ5.pcap


In [2]:
capture = read_pcap(str(PCAP))
transactions = build_transactions(capture)
findings = analyze_transactions(transactions)
comparisons = build_comparisons(transactions, findings)
iid = next((m.group(1) for tx in transactions if (m := re.match(r"^/(\d{15,})/", tx.target))), "")
short = lambda s, n=72: (s.replace("/" + iid, "/{id}") if iid else s)[:n]
print(f"packets={capture.packets} flows={len(capture.flows)} transactions={len(transactions)} duration={capture.duration:.3f}s xss={len(findings)}")
attack_ids = {f.transaction.index for f in findings}
print(f"\n{'#':>3} {'method':<6} {'url':<72} {'status':>6} {'wire/dec':>12} {'xss':>4}")
for tx in transactions:
    r = tx.response
    lengths = f"{r.body_wire_length}/{len(r.body_decoded)}" if r else "0/0"
    print(f"{tx.index:>3} {tx.method:<6} {short(tx.target):<72} {tx.status:>6} {lengths:>12} {'XSS' if tx.index in attack_ids else '':>4}")

packets=165 flows=28 transactions=14 duration=5.012s xss=4

  # method url                                                                      status     wire/dec  xss
  1 GET    /start                                                                      200      444/681     
  2 GET    /{id}                                                                       302      172/205     
  3 GET    /{id}/                                                                      200    1231/3480     
  4 GET    /{id}/login?uid=cheddar&pw=orange                                           200    1447/4350     
  5 GET    /{id}/feed.gtl?uid=cheddar                                                  200      135/146     
  6 GET    /{id}/snippets.gtl?uid=cheddar                                              200    1296/3748     
  7 GET    /{id}/                                                                      200    1447/4350     
  8 GET    /{id}/feed.gtl?uid=%3Cscript%3Ealert%28%27XSS%27%29%3C%2F

In [3]:
tx8 = next(t for t in transactions if t.index == 8)
print(f"gzip: {tx8.response.body_wire_length} -> {len(tx8.response.body_decoded)} bytes")
print(tx8.response.text())

gzip: 69 -> 54 bytes

_feed((


[
"<script>alert('XSS')</script>"

]



))



In [4]:
for f in findings:
    tx = f.transaction
    print(f"\n[{tx.index}] {tx.method} {tx.target}")
    for i in f.injections:
        print(f"  injection: {i.location} {i.name} = {i.value}")
        if i.location in ("query", "path"):
            print(f"  raw: {i.raw_value()[:200]}")
    for r in f.reflections:
        if r.reflected:
            print(f"  reflection: {r.variant}; {r.context}; tx={r.transaction_index}\n  snippet: {r.snippet[:220]}")
f10 = next(f for f in findings if f.transaction.index == 10)
for r in f10.reflections:
    if r.reflected:
        print(f"\nstored XSS: tx={r.transaction_index}\n  {r.snippet[:220]}")
payload = "<img src=\"nonexistent.jpg\" onerror=\"alert('XSS')\">"
print("payload in 11/12:", *(payload in next(t for t in transactions if t.index == n).response.text() for n in (11, 12)))


[8] GET /616469580724350452632383255415723265857/feed.gtl?uid=%3Cscript%3Ealert%28%27XSS%27%29%3C%2Fscript%3E
  injection: query uid = <script>alert('XSS')</script>
  raw: %3Cscript%3Ealert%28%27XSS%27%29%3C%2Fscript%3E
  reflection: raw; HTML-тело: внедрён новый тег <script>; tx=8
  snippet: _feed(( [ "<script>alert('XSS')</script>" ] ))

[9] GET /616469580724350452632383255415723265857/%3Cscript%3Ealert%28%27XSS%27%29%3C/script%3E
  injection: path PATH = /616469580724350452632383255415723265857/<script>alert('XSS')</script>
  raw: %2F616469580724350452632383255415723265857%2F%3Cscript%3Ealert%28%27XSS%27%29%3C%2Fscript%3E
  reflection: raw; HTML-тело: внедрён новый тег <script>; tx=9
  snippet: ut</a> </span> </div> <div class='message'>Invalid request: /<script>alert('XSS')</script></div> </body> </html>

[10] GET /616469580724350452632383255415723265857/newsnippet2?snippet=%3Cimg%20src%3D%22nonexistent.jpg%22%20onerror%3D%22alert%28%27XSS%27%29%22%3E
  injection: query snippet = 

In [5]:
for c in comparisons:
    a, b = c.attack.transaction, c.baseline
    print(f"\nattack #{a.index}, baseline #{b.index if b else '-'}, same={c.same_endpoint}")
    print("request:\n" + "\n".join(f"  {x}" for x in c.request_changes))
    print("response:\n" + "\n".join(f"  {x}" for x in c.response_changes))
    print("diff:\n" + "\n".join(f"  {x}" for x in c.diff_excerpt))


attack #8, baseline #5, same=True
request:
  URL: /616469580724350452632383255415723265857/feed.gtl?uid=cheddar -> /616469580724350452632383255415723265857/feed.gtl?uid=%3Cscript%3Ealert%28%27XSS%27%29%3C%2Fscript%3E
  query 'uid': 'cheddar' -> '<script>alert('XSS')</script>'
response:
  статус: 200 -> 200
  Content-Type: 'text/html' -> 'text/html'
  длина тела (в сети): 135 -> 69 байт; после распаковки: 146 -> 54 байт
  Content-Encoding: gzip
  payload отражён (raw); контекст: HTML-тело: внедрён новый тег <script>
diff:
  --- обычный запрос
  +++ XSS-запрос
  @@ -4,9 +4,5 @@
  -"Cheddar Mac"
  -
  -,"Gruyere is the cheesiest application on the web."
  -
  -,"I wonder if there are any security holes in this...."
  +"<script>alert('XSS')</script>"

attack #9, baseline #7, same=False
request:
  URL: /616469580724350452632383255415723265857/ -> /616469580724350452632383255415723265857/%3Cscript%3Ealert%28%27XSS%27%29%3C/script%3E
  path 'PATH': '/616469580724350452632383255415723265857/'